<a href="https://colab.research.google.com/github/TaherBenAfia/Fly2/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
rule = 'if a page is stale and still visible, check its ctr and engagement to pick a refresh action'
reason_code_list = ['stale_content', 'high_visibility', 'low_ctr', 'low_engagement']
action_list = ['refresh_and_review_ctr', 'refresh_and_review_engagement', 'refresh', 'monitor']
print(rule)
print(reason_code_list)
print(action_list)

if a page is stale and still visible, check its ctr and engagement to pick a refresh action
['stale_content', 'high_visibility', 'low_ctr', 'low_engagement']
['refresh_and_review_ctr', 'refresh_and_review_engagement', 'refresh', 'monitor']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import json
df['stale'] = (df['days_since_last_update'] >= 180).astype(int)
df['visible'] = (df['impressions_90d'] >= 500).astype(int)
df['low_ctr'] = (df['ctr'] < 1.0).astype(int)
df['low_engagement'] = (df['engagement_rate'] < 20).astype(int)

def pick_action(row):
    if row['stale'] and row['visible'] and row['low_ctr']:
        return 'refresh_and_review_ctr'
    if row['stale'] and row['visible'] and row['low_engagement']:
        return 'refresh_and_review_engagement'
    if row['stale'] and row['visible']:
        return 'refresh'
    return 'monitor'

def reason_codes(row):
    codes = []
    if row['stale']: codes.append('stale_content')
    if row['visible']: codes.append('high_visibility')
    if row['low_ctr']: codes.append('low_ctr')
    if row['low_engagement']: codes.append('low_engagement')
    return ', '.join(codes)

df['baseline_score'] = df['stale'] * df['visible'] * df['impressions_90d']
df['suggested_action'] = df.apply(pick_action, axis=1)
df['reason_codes'] = df.apply(reason_codes, axis=1)

os.makedirs('work/outputs', exist_ok=True)
cols = ['content_id','client_id','baseline_score','suggested_action','reason_codes',
        'impressions_90d','days_since_last_update','avg_position','ctr','engagement_rate','trend_direction']
queue = df.sort_values('baseline_score', ascending=False)
queue[cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(df['suggested_action'].value_counts())
metrics = {
    'rows_scored': int(len(df)),
    'action_counts': df['suggested_action'].value_counts().to_dict(),
    'top_queue_score': float(queue['baseline_score'].max()),
    'declining_rate_in_top20': float(queue.head(20)['trend_direction'].eq('down').mean()),
    'rule': 'stale(>=180d) x visible(>=500 impr) -> ctr/engagement branch'
}
with open('work/outputs/baseline_action_score_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(metrics)

suggested_action
monitor                   29983
refresh_and_review_ctr       17
Name: count, dtype: int64
{'rows_scored': 30000, 'action_counts': {'monitor': 29983, 'refresh_and_review_ctr': 17}, 'top_queue_score': 61678.0, 'declining_rate_in_top20': 0.85, 'rule': 'stale(>=180d) x visible(>=500 impr) -> ctr/engagement branch'}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
caveats = {
    'low_ctr': 'wrong if a SERP feature is stealing clicks rather than the page underperforming',
    'low_engagement': 'wrong if GA4 engagement tracking is broken or blocked for this page',
    'stale_content': 'wrong if the page was reviewed but the update was not logged',
    'high_visibility': 'wrong if impressions include bot or spam traffic'
}

top20 = queue.head(20).copy()
top20['confidence'] = np.where(top20['impressions_90d'] >= 5000, 'high',
                        np.where(top20['impressions_90d'] >= 500, 'medium', 'low'))
top20['what_would_make_it_wrong'] = top20['reason_codes'].apply(
    lambda codes: '; '.join(caveats[c] for c in codes.split(', ') if c in caveats)
)
top20[['content_id','suggested_action','reason_codes','confidence','what_would_make_it_wrong']]

,content_id,suggested_action,reason_codes,confidence,what_would_make_it_wrong
16751,content_cf56e2e2e282,refresh_and_review_ctr,"stale_content, high_visibility, low_ctr, low_e...",high,wrong if the page was reviewed but the update ...
16514,content_7368877ea310,refresh_and_review_ctr,"stale_content, high_visibility, low_ctr, low_e...",high,wrong if the page was reviewed but the update ...
7021,content_1bfaa38ff26c,refresh_and_review_ctr,"stale_content, high_visibility, low_ctr, low_e...",high,wrong if the page was reviewed but the update ...
21268,content_0a91db491d14,refresh_and_review_ctr,"stale_content, high_visibility, low_ctr, low_e...",high,wrong if the page was reviewed but the update ...
11489,content_5feee3994adb,refresh_and_review_ctr,"stale_content, high_visibility, low_ctr, low_e...",high,wrong if the page was reviewed but the update ...
12045,content_c2d929d83eaa,refresh_and_review_ctr,"stale_content, high_visibility, low_ctr, low_e...",high,wrong if the page was reviewed but the update ...
698,content_b16bd7307b39,refresh_and_review_ctr,"stale_content, high_visibility, low_ctr, low_e...",medium,wrong if the page was reviewed but the update ...
5327,content_fe16a55cd13d,refresh_and_review_ctr,"stale_content, high_visibility, low_ctr, low_e...",medium,wrong if the page was reviewed but the update ...
26810,content_ecb6215e79fd,refresh_and_review_ctr,"stale_content, high_visibility, low_ctr",medium,wrong if the page was reviewed but the update ...
20837,content_928af3e22c80,refresh_and_review_ctr,"stale_content, high_visibility, low_ctr, low_e...",medium,wrong if the page was reviewed but the update ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
assert 'trend_direction' not in ['stale','visible','low_ctr','low_engagement','baseline_score']
assert 'trend_pct' not in ['stale','visible','low_ctr','low_engagement','baseline_score']

weak_picks = queue[(queue['trend_direction'].isin(['up','stable'])) &
                    (queue['avg_position'] <= 30) &
                    (queue['suggested_action'] != 'monitor')]
weak_picks[['content_id','avg_position','trend_direction','suggested_action','ctr','engagement_rate']]
weak_pick_metrics = {'weak_pick_count': int(len(weak_picks))}
with open('work/outputs/weak_picks_metrics.json', 'w') as f:
    json.dump(weak_pick_metrics, f, indent=2)
print(weak_pick_metrics)

{'weak_pick_count': 1}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.